# DroneZ GRPO Colab

This notebook validates the DroneZ training interface and prepares a real TRL/GRPO run path. Dry-run cells do not claim real training happened.

In [ ]:
import os
import pathlib
import subprocess
import sys

REPO_URL = os.environ.get("DRONEZ_REPO_URL", "https://github.com/Hollenite/DroneZ.git")
WORKDIR = pathlib.Path("/content/DroneZ")

if not WORKDIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(WORKDIR)], check=True)

os.chdir(WORKDIR)
print("Working directory:", pathlib.Path.cwd())

## Install dependencies

The train extra installs TRL/Transformers dependencies. Unsloth can be installed separately if the current Colab image and GPU support it.

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".[train]"], check=True)

INSTALL_UNSLOTH = False
if INSTALL_UNSLOTH:
    subprocess.run([sys.executable, "-m", "pip", "install", "unsloth"], check=True)

## Dry-run first

This checks the actual DroneZ prompt/action/reward interface and writes reference artifacts. It is not real model training.

In [ ]:
subprocess.run([
    sys.executable,
    "scripts/train_grpo_colab.py",
    "--dry-run",
    "--model", "Qwen/Qwen2.5-0.5B-Instruct",
    "--tasks", "easy,medium,demo",
    "--output-dir", "artifacts/training",
], check=True)

## Optional real training gate

Set `RUN_REAL_TRAINING = True` only when a GPU runtime is selected and you are ready to run TRL/GRPO. The script will fail clearly if TRL dependencies are missing.

In [ ]:
RUN_REAL_TRAINING = False

if RUN_REAL_TRAINING:
    subprocess.run([
        sys.executable,
        "scripts/train_grpo_colab.py",
        "--model", "Qwen/Qwen2.5-0.5B-Instruct",
        "--tasks", "easy,medium,demo",
        "--output-dir", "artifacts/training",
    ], check=True)
else:
    print("Real GRPO training skipped. Dry-run artifacts are present; do not claim trained-model improvement yet.")

## Regenerate evaluation artifacts

These commands refresh deterministic evaluation artifacts for the README and replay UI.

In [ ]:
subprocess.run([sys.executable, "scripts/evaluate_policies.py"], check=True)
subprocess.run([sys.executable, "scripts/generate_demo_trace.py", "--task", "demo", "--policy", "all"], check=True)
subprocess.run([sys.executable, "scripts/generate_plots.py"], check=True)

## Zip artifacts for download

Download this zip and add real training outputs back to the repo only after a real run.

In [ ]:
import shutil

zip_path = shutil.make_archive("dronez_artifacts", "zip", "artifacts")
print("Wrote", zip_path)